# Amaliyot 9. Giperparametrlarni tanlash

9-interaktiv laboratoriyaga («Giperparametrlarni tanlash») hamroh. Kurs mavzusi: Grid Search va Random Search

Daftar oʻz-oʻzidan yetarli: birinchi katak `common.py` faylini oʻquv korpusi va yordamchi funksiyalar bilan yaratadi. Kataklarni yuqoridan pastga ishga tushiring. Korpus oʻquv (shablon) boʻlib, haqiqiy masalada oʻz CSV faylingizni quyidagi ustunlar bilan qoʻying: `text` va `label`.

In [ ]:
%%writefile common.py
"""Matnlarni tasniflash va klasterlash amaliyotlari uchun umumiy funksiyalar."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "sport": ("oʻyin jamoa gol oʻyinchi murabbiy chempionat turnir gʻalaba magʻlubiyat mavsum stadion muxlis hakam hisob "
              "himoyachi hujumchi darvozabon kubok liga final transfer klub zarba penalti musobaqa sportchi "
              "rekord medal olimpiada mashq durang").split(),
    "siyosat": ("qonun deputat saylov partiya ovoz prezident vazir hukumat parlament islohot byudjet "
                "sanksiya nomzod koalitsiya vazirlik farmon tuzatish loyiha muxolifat referendum diplomatiya "
                "hokim konstitutsiya majlis komissiya muzokara sammit saylovchi mandat vakolat").split(),
    "oshxona": ("shoʻrva retsept tuz goʻsht pishirmoq duxovka sabzavot tova qozon xamir shakar ziravor qaynatmoq qovurmoq "
                "dimlamoq sous bulyon salat oshpaz taom masalliq yogʻ un tuxum sut sarimsoq piyoz garmdori "
                "kartoshka shirinlik pishiriq marinad").split(),
    "texnologiya": ("protsessor dastur ilova smartfon neyrotarmoq algoritm maʼlumot server yangilanish dasturchi "
                    "kompyuter xotira ekran kamera brauzer tarmoq internet bulut kod platforma foydalanuvchi qurilma "
                    "chip batareya noutbuk dasturiy videokarta tizim interfeys shifrlash").split(),
}
NEUTRAL = ("bugun juda yangi aytishadi keyin ham mumkin edi boʻldi maʼlum qilindi kecha yaqinda hali hammasi "
           "bir nechta muhim birinchi").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """4 mavzudagi qisqa «hujjatlar»dan iborat oʻquv korpusi.
    noise: oʻz mavzusidan boʻlmagan soʻzlar ulushi (neytral yoki begona). noise qancha katta boʻlsa, masala shuncha qiyin."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """TF-IDF siyrak matritsasini tekislikka proyeksiyalash (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Har bir sinf uchun eng katta vaznli soʻzlarni chiqaradi (chiziqli modellar)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))


In [ ]:
from common import *

# Oʻquv korpusi: 4 ta mavzu. noise — "begona" soʻzlar ulushi (qancha koʻp boʻlsa, shuncha qiyin).
df = make_corpus(n_per_class=150, noise=0.55, seed=0)
# Oʻquv korpusi oʻrniga oʻzingiznikini qoʻying: df = pd.read_csv("texts.csv")  # ustunlar: text, label
print(df.label.value_counts().to_dict())
df.head()

In [ ]:
import time
from scipy.stats import loguniform
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split

Xtr, Xte, ytr, yte = train_test_split(df.text, df.label, test_size=0.25, stratify=df.label, random_state=0)
pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=1000))])

## 1. Grid Search: toʻrni sinab chiqish

In [ ]:
grid = {"tfidf__ngram_range": [(1, 1), (1, 2)], "tfidf__min_df": [1, 3], "clf__C": [0.01, 0.1, 1, 10, 100]}
t0 = time.time()
gs = GridSearchCV(pipe, grid, cv=3, scoring="f1_macro", n_jobs=-1).fit(Xtr, ytr)
print("Grid:", round(time.time() - t0, 1), "s | kombinatsiyalar:", len(gs.cv_results_["params"]))
print("Eng yaxshi parametrlar:", gs.best_params_, "| CV:", round(gs.best_score_, 3))

## 2. Random Search: xuddi shu sinovlar byudjeti, tasodifiy kombinatsiyalar

In [ ]:
dist = {"tfidf__ngram_range": [(1, 1), (1, 2)], "tfidf__min_df": [1, 2, 3, 5], "clf__C": loguniform(1e-2, 1e2)}
t0 = time.time()
rs = RandomizedSearchCV(pipe, dist, n_iter=10, cv=3, scoring="f1_macro", n_jobs=-1, random_state=0).fit(Xtr, ytr)
print("Random:", round(time.time() - t0, 1), "s | kombinatsiyalar:", 10)
print("Eng yaxshi parametrlar:", rs.best_params_, "| CV:", round(rs.best_score_, 3))

In [ ]:
res = pd.DataFrame(gs.cv_results_)
res["C"] = res["param_clf__C"].astype(float)
plt.scatter(res["C"], res["mean_test_score"], label="grid")
r2 = pd.DataFrame(rs.cv_results_); r2["C"] = r2["param_clf__C"].astype(float)
plt.scatter(r2["C"], r2["mean_test_score"], marker="x", label="random")
plt.xscale("log"); plt.xlabel("C"); plt.ylabel("F1 macro (CV)"); plt.legend(); plt.title("Har bir qidiruv C ning qaysi qiymatlarini tekshirdi"); plt.show()

## 3. Tegilmagan testda halol baholash

In [ ]:
from sklearn.metrics import f1_score
for name, s in [("grid", gs), ("random", rs)]:
    print(name, "test F1:", round(f1_score(yte, s.predict(Xte), average="macro"), 3))

## Topshiriqlar

1. Qiymatini oshiring: `n_iter` — tasodifiy qidiruv uchun 40 gacha. Eng yaxshi CV baho qanday oʻsadi?
2. Qidiruvga boshqa modellarni qoʻshing (masalan, `LinearSVC`) parametrlar roʻyxati orqali `clf`.
3. Sinab koʻring `HalvingRandomSearchCV` (`from sklearn.experimental import enable_halving_search_cv`). U nimasi bilan farq qiladi?